In [1]:
import os
import sys
import contextlib
import numpy as np
import json
from sklearn.covariance import graphical_lasso
from matplotlib import pyplot as plt

from method import EM_algorithm as em, tlasso
from simulation import simulation_data_generator as dg

COLOR_ASYM = "#2a78d6"
COLOR_EM_DIAG = "#f5239a"
COLOR_GGM = "#e34948"
COLOR_T = "#f5a623"
COLOR_TS = "#23f57e"
COLOR_CHANCE = "#c3c2b7"


def offdiag_max(S):
    """max_{i != j} |S_ij|. The smallest rho that zeroes every off-diagonal."""
    A = np.abs(np.asarray(S, dtype=float)).copy()
    np.fill_diagonal(A, 0.0)
    rmax = float(A.max())
    if not np.isfinite(rmax) or rmax <= 0:
        raise ValueError("rho_max is not positive; check the input matrix.")
    return rmax


def make_rho_grid(rho_max, n_rho=50, max_ratio = 1, min_ratio=0.05):
    """Log-spaced, DESCENDING grid on [min_ratio * rho_max, rho_max].
 
    Log spacing because edge count is roughly geometric in rho: linear
    spacing wastes most points in the dense end where the ROC curve barely
    moves. Descending so warm starts run sparse -> dense, which is the
    numerically stable direction when p > n.
    """
    return np.logspace(np.log10(max_ratio * rho_max), np.log10(min_ratio * rho_max), n_rho)


def pilot_rho_max(
    Y, algorithm, rho, algorithm_kwargs=None, rho_name="rho", S_key="S_tau",
):
    """rho_max for one method on one replicate: fit at `rho`, read off its S.

    The EM M-step penalizes the working covariance S_tau, not cov(Y): S_tau is
    tau-weighted, skew-corrected and (for MWGP) Monte-Carlo averaged, so its
    off-diagonal scale is method-specific. Building every method's grid from
    cov(Y) therefore starts each path at a different point along its own
    regularization range -- some methods begin already empty, others begin
    dense, and the index-wise average across replicates mixes those.

    S_tau also depends on rho, so there is no rho-free version of it to
    calibrate against. The pilot fit at the theoretical rho = sqrt(log p / n)
    is the reference point: run the EM there to convergence, take
    max_{i != j} |S_ij| of the converged S_tau, and use that as rho_max.
    """
    kwargs = {} if algorithm_kwargs is None else dict(algorithm_kwargs)
    print(f"  pilot fit of {algorithm.__name__} at rho={rho:.5f}")
    res = algorithm(Y, **{rho_name: rho}, **kwargs)
    rho_max = offdiag_max(res[S_key])
    print(f"  -> rho_max = max|S_ij| (off-diag) = {rho_max:.5f}")
    return rho_max


def edge_confusion(Theta_hat, true_pos_mask, true_neg_mask, tol=1e-8):
    iu = np.triu_indices(Theta_hat.shape[0], k=1)
    est_edges = np.abs(Theta_hat[iu]) > tol
    tp_rate = np.sum(est_edges & true_pos_mask) / true_pos_mask.sum()
    fp_rate = np.sum(est_edges & true_neg_mask) / true_neg_mask.sum()
    return fp_rate, tp_rate


@contextlib.contextmanager
def stream_to(path, also_stderr=True):
    """Send print()/warnings to `path` instead of the cell output.

    buffering=1 is the point: line buffering means the file fills as the run
    goes, so `Get-Content <path> -Wait` follows it live. On an exception the
    streams are restored before the traceback renders, so errors still show
    up in the notebook.
    """
    f = open(path, "w", buffering=1, encoding="utf-8")
    old_out, old_err = sys.stdout, sys.stderr
    sys.stdout = f
    if also_stderr:
        sys.stderr = f
    try:
        yield f
    finally:
        sys.stdout, sys.stderr = old_out, old_err
        f.close()


def roc_curve_em(
    Y, rho_grid, algorithm, true_pos_mask, true_neg_mask,
    algorithm_kwargs=None, rho_name="rho", theta_key="Theta",
):
    """Re-run the whole EM at each rho, warm-started, sparsest first.

    `algorithm_kwargs` holds whatever extra arguments the given algorithm takes
    (e.g. nu/n_iter for run_tlasso, n_burn/n_keep for run_em_MWGP); `rho_name`
    and `theta_key` cover algorithms that name the penalty or the returned
    precision matrix differently.

    Each fit is initialized at the previous rho's mu and Theta. Only those two
    are carried: nu/eta are tail-shape nuisance parameters, and handing them
    forward lets nu ratchet down across the sweep until lam = -2/nu - 0.5 is
    negative enough to overflow the Bessel terms in gig_moment.

    These EM objectives are not convex, so warm starting changes the estimator
    and not just the runtime: every point on the curve depends on the whole
    path prefix, and the sweep must stay sparse -> dense for the results to be
    reproducible.
    """
    kwargs = {} if algorithm_kwargs is None else dict(algorithm_kwargs)
    fp = np.empty(len(rho_grid))
    tp = np.empty(len(rho_grid))
    Theta_prev = np.eye(Y.shape[1])
    prev = None
    print(f"Running {algorithm.__name__} over {len(rho_grid)} rho values...")
    for i in range(len(rho_grid)):
        print(f"  rho={rho_grid[i]:.5f} ({i+1}/{len(rho_grid)})")

        res = algorithm(
            Y, **{rho_name: rho_grid[i]}, **kwargs,
            **({"init": prev} if prev is not None else {}),
        )
        Theta_hat = res[theta_key]
        if np.all(np.isfinite(Theta_hat)):
            prev = {"mu": res["mu"], "Theta": Theta_hat}

        Theta_prev = Theta_hat
        fp[i], tp[i] = edge_confusion(Theta_hat, true_pos_mask, true_neg_mask)
        print(fp[i], tp[i])
    return fp, tp


def roc_curve_em_autogrid(
    Y, algorithm, true_pos_mask, true_neg_mask, pilot_rho, n_rho,
    min_ratio=0.05, max_ratio=1, algorithm_kwargs=None, rho_name="rho",
    theta_key="Theta", S_key="S_tau",
):
    """pilot_rho_max -> make_rho_grid -> roc_curve_em, for one method/replicate.

    Returns (fp, tp, rho_grid); the grid is method- and replicate-specific and
    has to be kept, since the theoretical rho no longer sits at a common index.

    The pilot fit is NOT reused as a warm start: the sweep must run
    sparse -> dense from its own top end for the path to be reproducible, and
    the pilot sits in the middle of it.
    """
    rho_max = pilot_rho_max(
        Y, algorithm, pilot_rho, algorithm_kwargs=algorithm_kwargs,
        rho_name=rho_name, S_key=S_key,
    )
    if pilot_rho > rho_max:
        print(f"  [warn] pilot rho {pilot_rho:.5f} > rho_max {rho_max:.5f}: "
              f"it falls off the top of this method's grid")
    rho_grid = make_rho_grid(
        rho_max, n_rho=n_rho, max_ratio=max_ratio, min_ratio=min_ratio,
    )
    print(f"  rho grid: {rho_grid}")
    fp, tp = roc_curve_em(
        Y, rho_grid, algorithm, true_pos_mask, true_neg_mask,
        algorithm_kwargs=algorithm_kwargs, rho_name=rho_name,
        theta_key=theta_key,
    )
    return fp, tp, rho_grid


def roc_curve_glasso(Y, rho_grid, true_pos_mask, true_neg_mask, glasso_kwargs=None):
    """roc_curve_full_em analogue for the naive Gaussian glasso baseline.

    sklearn's graphical_lasso takes an empirical covariance rather than Y,
    names the penalty `alpha`, and returns (covariance, precision), so it needs
    its own loop.
    """
    kwargs = {} if glasso_kwargs is None else dict(glasso_kwargs)
    S = np.cov(Y, rowvar=False) + 1e-10 * np.eye(Y.shape[1])
    fp = np.empty(len(rho_grid))
    tp = np.empty(len(rho_grid))
    
    p = Y.shape[1]
    Theta_prev = np.eye(p)
    print(f"Running graphical_lasso over {len(rho_grid)} rho values...")
    for i in range(len(rho_grid)):
        print(f"  rho={rho_grid[i]:.5f} ({i+1}/{len(rho_grid)})")
        try:
            _, Theta_hat = graphical_lasso(S + rho_grid[i] * np.eye(p), alpha=rho_grid[i], **kwargs)
        except Exception:
            Theta_hat = Theta_prev
        Theta_prev = Theta_hat
        fp[i], tp[i] = edge_confusion(Theta_hat, true_pos_mask, true_neg_mask)
        
    
    return fp, tp


def roc_curve_glasso_autogrid(
    Y, true_pos_mask, true_neg_mask, n_rho, min_ratio=0.05, max_ratio=1,
    glasso_kwargs=None,
):
    """roc_curve_em_autogrid for the Gaussian baseline; no pilot fit needed.

    glasso penalizes the empirical covariance itself, which does not depend on
    rho, so its rho_max is available in closed form -- the pilot fit would
    return the same S it was handed.
    """
    S = np.cov(Y, rowvar=False)
    rho_max = offdiag_max(S)
    print(f"  graphical_lasso rho_max = max|S_ij| (off-diag) = {rho_max:.5f}")
    rho_grid = make_rho_grid(
        rho_max, n_rho=n_rho, max_ratio=max_ratio, min_ratio=min_ratio,
    )
    print(f"  rho grid: {rho_grid}")
    fp, tp = roc_curve_glasso(
        Y, rho_grid, true_pos_mask, true_neg_mask, glasso_kwargs=glasso_kwargs,
    )
    return fp, tp, rho_grid


def auc_from_curve(fp, tp):
    order = np.argsort(fp)
    fp_sorted = np.concatenate([[0.0], fp[order], [1.0]])
    tp_sorted = np.concatenate([[0.0], tp[order], [1.0]])
    return np.trapezoid(tp_sorted, fp_sorted)


def mean_roc(fp, tp, n_points=201):
    """Vertical average of the replicate ROC curves (Fawcett 2006, Sec. 6.1).

    Averaging fp and tp down a column would be threshold averaging, which needs
    column i to be the same threshold in every row. It is not: every replicate
    has its own rho grid built from its own rho_max, so column i is a different
    rho in every row. Averaging it smears the curve horizontally, and where the
    curves are convex the mean point can land below all of them.

    Interpolating each replicate's TPR onto a common FPR grid and averaging
    vertically needs no correspondence between the rho grids at all. Returns
    (fp_common, tp_mean, tp_se).
    """
    fp = np.asarray(fp, dtype=float)
    tp = np.asarray(tp, dtype=float)
    n_rep = fp.shape[0]
    fp_common = np.linspace(0.0, 1.0, n_points)
    tp_interp = np.empty((n_rep, n_points))
    for r in range(n_rep):
        # Anchor at (0,0) and (1,1): the path stops at min_ratio * rho_max, so
        # it need not reach either corner on its own, and np.interp would
        # otherwise extrapolate flat from whatever the endpoints happen to be.
        x = np.concatenate([[0.0], fp[r], [1.0]])
        y = np.concatenate([[0.0], tp[r], [1.0]])
        order = np.argsort(x, kind="stable")
        x, y = x[order], y[order]
        # np.interp needs strictly increasing x. Ties are one FPR reached at
        # several rho; keep the best TPR there, i.e. the upper envelope.
        x_u, first = np.unique(x, return_index=True)
        y_u = np.maximum.reduceat(y, first)
        tp_interp[r] = np.interp(fp_common, x_u, y_u)
    tp_se = (
        tp_interp.std(axis=0, ddof=1) / np.sqrt(n_rep) if n_rep > 1
        else np.zeros(n_points)
    )
    return fp_common, tp_interp.mean(axis=0), tp_se


def operating_point_at_rho(fp, tp, rho_grids, rho):
    """Mean (FPR, TPR) at the grid point nearest `rho` in each replicate.

    The one place threshold averaging is still the right thing: a rho value is
    comparable across replicates and across methods even though a grid index is
    not. Nearest is taken in log rho, matching the log spacing of the grid.
    """
    rho_grids = np.asarray(rho_grids, dtype=float)
    i = np.argmin(np.abs(np.log(rho_grids) - np.log(rho)), axis=1)
    r = np.arange(rho_grids.shape[0])
    return float(np.asarray(fp)[r, i].mean()), float(np.asarray(tp)[r, i].mean())

In [2]:
filename = "fifty_simulations"
num_simulations=50
p=100
n=50
num_rho=30

In [3]:
Theta_true = dg.make_true_theta(p)

iu = np.triu_indices(p, k=1)
true_pos_mask = Theta_true[iu] != 0
true_neg_mask = ~true_pos_mask

theoretical_rho = np.sqrt(np.log(p) / n)
# make a evenly spaced grid centered at theoretical rho with half length width

print(f"theoretical rho = sqrt(log({p}) / {n}) = {theoretical_rho:.5g}")

theoretical rho = sqrt(log(100) / 50) = 0.30349


In [4]:
fp_mwgp = np.empty((num_simulations, num_rho))
tp_mwgp = np.empty((num_simulations, num_rho))
fp_em_diag = np.empty((num_simulations, num_rho))
tp_em_diag = np.empty((num_simulations, num_rho))
fp_ggm = np.empty((num_simulations, num_rho))
tp_ggm = np.empty((num_simulations, num_rho))
fp_t = np.empty((num_simulations, num_rho))
tp_t = np.empty((num_simulations, num_rho))
fp_ts = np.empty((num_simulations, num_rho))
tp_ts = np.empty((num_simulations, num_rho))
auc_mwgp = np.empty(num_simulations)
auc_em_diag = np.empty(num_simulations)
auc_ggm = np.empty(num_simulations)
auc_t = np.empty(num_simulations)
auc_ts = np.empty(num_simulations)
# One grid per method per replicate: rho_max is read off that method's own
# converged S_tau, so index i is the same relative position on the path
# (min_ratio ... 1 of rho_max) but not the same rho across methods.
rho_grids_mwgp = np.empty((num_simulations, num_rho))
rho_grids_em_diag = np.empty((num_simulations, num_rho))
rho_grids_ggm = np.empty((num_simulations, num_rho))
rho_grids_t = np.empty((num_simulations, num_rho))
rho_grids_ts = np.empty((num_simulations, num_rho))

In [ ]:
# Output goes to roc_run.log so it does not flood the cell.
# Follow it live:  Get-Content roc_run.log -Wait -Tail 20
#
# Each method gets its own rho grid in each replicate: fit at the theoretical
# rho first, let it converge, then take rho_max = max_{i != j} |S_ij| off the
# converged S_tau. See pilot_rho_max for why a shared cov(Y) grid is wrong here.
kwargs_t = {"n_iter": 200, "verbose": False}
kwargs_ts = {"n_iter": 200, "verbose": False}
kwargs_em_diag = {"n_iter": 200, "verbose": False}
kwargs_mwgp = {
    "n_iter": 50,
    "verbose": True,
    "warning": True,
    "mcmc_samples": 100,
    "mcmc_thin": 1,
    "mcmc_warmup": 10,
    "proposal": "gig",
}

with stream_to("roc_run.log"):
    for sim in range(num_simulations):
        print(f"Replicate {sim + 1}/{num_simulations}")

        # Generate data from an independent model (noisy skewed Gaussian)
        Y = dg.simulate_contaminated_normal_data(n, p, Theta_true)

        # Classical t-distribution model (run_tlasso)
        fp_t[sim], tp_t[sim], rho_grids_t[sim] = roc_curve_em_autogrid(
            Y, tlasso.run_tlasso, true_pos_mask, true_neg_mask,
            pilot_rho=theoretical_rho, n_rho=num_rho, min_ratio=0.10,
            algorithm_kwargs=kwargs_t,
        )
        auc_t[sim] = auc_from_curve(fp_t[sim], tp_t[sim])

        # Alternative t-distribution model (run_tstar_varlasso)
        fp_ts[sim], tp_ts[sim], rho_grids_ts[sim] = roc_curve_em_autogrid(
            Y, tlasso.run_tstar_varlasso, true_pos_mask, true_neg_mask,
            pilot_rho=theoretical_rho, n_rho=num_rho, min_ratio=0.10,
            algorithm_kwargs=kwargs_ts,
        )
        auc_ts[sim] = auc_from_curve(fp_ts[sim], tp_ts[sim])
    
        # Asymmetric Alternative t-distribution model (EM_DIAGONAL)
        fp_em_diag[sim], tp_em_diag[sim], rho_grids_em_diag[sim] = roc_curve_em_autogrid(
            Y, em.run_em_diagonal, true_pos_mask, true_neg_mask,
            pilot_rho=theoretical_rho, n_rho=num_rho, min_ratio=0.10,
            algorithm_kwargs=kwargs_em_diag,
        )
        auc_em_diag[sim] = auc_from_curve(fp_em_diag[sim], tp_em_diag[sim])
    
        # Asymmetric Alternative t-distribution model (EM_MWGP)
        fp_mwgp[sim], tp_mwgp[sim], rho_grids_mwgp[sim] = roc_curve_em_autogrid(
            Y, em.run_em_MWGP, true_pos_mask, true_neg_mask,
            pilot_rho=theoretical_rho, n_rho=num_rho, min_ratio=0.10,
            algorithm_kwargs=kwargs_mwgp,
        )
        auc_mwgp[sim] = auc_from_curve(fp_mwgp[sim], tp_mwgp[sim])

        # Naive Gaussian graphical lasso baseline
        fp_ggm[sim], tp_ggm[sim], rho_grids_ggm[sim] = roc_curve_glasso_autogrid(
            Y, true_pos_mask, true_neg_mask, n_rho=num_rho, min_ratio=0.10,
            glasso_kwargs={"max_iter": 2000, "verbose": False},
        )
        auc_ggm[sim] = auc_from_curve(fp_ggm[sim], tp_ggm[sim])


In [ ]:
# Average the ROC curves across replicates and compute mean AUCs.
#
# Vertical averaging, not index-wise: the rho grids are replicate- and
# method-specific now, so a column of fp/tp is not a common threshold. See
# mean_roc. AUC is unaffected either way -- it is computed per replicate from
# that replicate's own curve and only then averaged.
fp_common, tp_mwgp_mean, tp_mwgp_se = mean_roc(fp_mwgp, tp_mwgp)
_, tp_em_diag_mean, tp_em_diag_se = mean_roc(fp_em_diag, tp_em_diag)
_, tp_ggm_mean, tp_ggm_se = mean_roc(fp_ggm, tp_ggm)
_, tp_t_mean, tp_t_se = mean_roc(fp_t, tp_t)
_, tp_ts_mean, tp_ts_se = mean_roc(fp_ts, tp_ts)

print(f"\n(p={p}, n={n}) over {num_simulations} replicates")
print(f"  Asymmetric model (MWGP): AUC = {auc_mwgp.mean():.3f} "
        f"(SE {auc_mwgp.std(ddof=1) / np.sqrt(num_simulations):.3f})")
print(f"  Asymmetric model (Diagonal): AUC = {auc_em_diag.mean():.3f} "
        f"(SE {auc_em_diag.std(ddof=1) / np.sqrt(num_simulations):.3f})")
print(f"  Classical t-model (TLASSO): AUC = {auc_t.mean():.3f} "
        f"(SE {auc_t.std(ddof=1) / np.sqrt(num_simulations):.3f})")
print(f"  Alternative t-model (TSTAR_VARLASSO): AUC = {auc_ts.mean():.3f} "
        f"(SE {auc_ts.std(ddof=1) / np.sqrt(num_simulations):.3f})")
print(f"  Naive Gaussian glasso (GGM): AUC = {auc_ggm.mean():.3f} "
        f"(SE {auc_ggm.std(ddof=1) / np.sqrt(num_simulations):.3f})")

METHODS = (
    ("Asym. model MWGP", fp_mwgp, tp_mwgp, tp_mwgp_mean, tp_mwgp_se, auc_mwgp,
     rho_grids_mwgp, COLOR_ASYM, "-"),
    ("Asym. diag. model", fp_em_diag, tp_em_diag, tp_em_diag_mean,
     tp_em_diag_se, auc_em_diag, rho_grids_em_diag, COLOR_EM_DIAG, "-"),
    ("Naive GGM", fp_ggm, tp_ggm, tp_ggm_mean, tp_ggm_se, auc_ggm,
     rho_grids_ggm, COLOR_GGM, "-."),
    ("Classical t-model", fp_t, tp_t, tp_t_mean, tp_t_se, auc_t,
     rho_grids_t, COLOR_T, "-."),
    ("Alternative t-model", fp_ts, tp_ts, tp_ts_mean, tp_ts_se, auc_ts,
     rho_grids_ts, COLOR_TS, "-."),
)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1, color=COLOR_CHANCE)
for name, fp_m, tp_m, tp_mean, tp_se, auc, grids, color, ls in METHODS:
    ax.plot(fp_common, tp_mean, color=color, linewidth=2, linestyle=ls,
            label=f"{name}, avg AUC={auc.mean():.3f}")
    ax.fill_between(fp_common, tp_mean - tp_se, tp_mean + tp_se,
                    color=color, alpha=0.15, linewidth=0)
    # The theoretical rho is a threshold, not a position on the averaged curve:
    # take it in each replicate and average those operating points. It can sit
    # slightly off the mean curve, which is honest -- that curve is a vertical
    # average and this point is not.
    fp_star, tp_star = operating_point_at_rho(fp_m, tp_m, grids, theoretical_rho)
    ax.plot(fp_star, tp_star, "o", color=color, zorder=5)
ax.plot([], [], "o", color="#52514e",
        label=r"$\rho=\sqrt{\log p\,/\,n}$" + f" = {theoretical_rho:.3g}")

ax.set_xlim(0, 1)
ax.set_ylim(0, 1.2)
ax.set_xlabel("false positive rate (1 - specificity)")
ax.set_ylabel("true positive rate (sensitivity)")
ax.set_title(f"Precision-matrix support recovery: p={p}, n={n}")
ax.legend(loc="lower right")
fig.tight_layout()

os.makedirs("results/simulations/roc", exist_ok=True)
filename = (
    "results/simulations/roc/roc" if filename is None
    else f"results/simulations/roc/{filename}"
)
fig.savefig(f"{filename}.pdf")


def method_result(fp, tp, tp_mean, tp_se, auc, grids):
    """Per-method block for the JSON dump.

    The raw per-replicate fp/tp and rho grids go in alongside the averaged
    curve: the mean curve is derived now (a vertical average on `fp_common`),
    and nothing downstream could rebuild it -- or re-average it differently --
    from the summary alone.
    """
    grids = np.asarray(grids)
    fp_star, tp_star = operating_point_at_rho(fp, tp, grids, theoretical_rho)
    return {
        "tp_mean": tp_mean.tolist(),
        "tp_se": tp_se.tolist(),
        "auc_mean": float(auc.mean()),
        "auc_se": float(auc.std(ddof=1) / np.sqrt(num_simulations)),
        "fp": np.asarray(fp).tolist(),
        "tp": np.asarray(tp).tolist(),
        "rho_grids": grids.tolist(),
        "rho_grid_mean": np.exp(np.log(grids).mean(axis=0)).tolist(),
        "theoretical_rho_point": [fp_star, tp_star],
    }


results = {
    "p": p,
    "n": n,
    "theoretical_rho": float(theoretical_rho),
    "fp_common": fp_common.tolist(),
    "asym_mwgp": method_result(
        fp_mwgp, tp_mwgp, tp_mwgp_mean, tp_mwgp_se, auc_mwgp, rho_grids_mwgp),
    "asym_em_diag": method_result(
        fp_em_diag, tp_em_diag, tp_em_diag_mean, tp_em_diag_se, auc_em_diag,
        rho_grids_em_diag),
    "ggm": method_result(
        fp_ggm, tp_ggm, tp_ggm_mean, tp_ggm_se, auc_ggm, rho_grids_ggm),
    "t": method_result(fp_t, tp_t, tp_t_mean, tp_t_se, auc_t, rho_grids_t),
    "ts": method_result(fp_ts, tp_ts, tp_ts_mean, tp_ts_se, auc_ts, rho_grids_ts),
}

with open(f"{filename}.json", "w") as f:
    json.dump(results, f)

In [ ]:
# Distribution of the simulated Y (the last replicate from the loop above).
# simulate_contaminated_normal_data replaces a fraction eps of individual
# ENTRIES with N(mu_star, contam_var) draws, so the contamination appears as a
# separate bump near mu_star rather than as a fattened tail.
from scipy.stats import norm, skew

sigma = np.linalg.inv(Theta_true)
mu_star = 2.5 * np.max(np.diag(sigma))          # mult * max(diag(sigma))

INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e6e5e0"
BAR, REF, MARK = "#2a78d6", "#52514e", "#e34948"

ncol = 5
nrow = int(np.ceil(p / ncol))
fig, axs = plt.subplots(nrow, ncol, figsize=(3.0 * ncol, 2.2 * nrow))
for j, ax in enumerate(axs.ravel()):
    if j >= p:
        ax.set_axis_off()
        continue
    yj = Y[:, j]
    ax.hist(yj, bins=60, density=True, color=BAR, edgecolor="white", linewidth=0.3)
    xs = np.linspace(yj.min(), yj.max(), 300)
    ax.plot(xs, norm.pdf(xs, yj.mean(), yj.std(ddof=1)),
            color=REF, linewidth=1.5, linestyle="--")
    ax.axvline(mu_star, color=MARK, linewidth=1.2)
    ax.set_title(f"$Y_{{{j + 1}}}$   skew {skew(yj):+.2f}", fontsize=9, color=INK)
    ax.tick_params(labelsize=7, colors=MUTED, length=3)
    ax.grid(axis="y", color=GRID, linewidth=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID)

# Legend at figure level: inside a panel it collides with the bars.
handles = [plt.Line2D([], [], color=REF, linestyle="--", linewidth=1.5),
           plt.Line2D([], [], color=MARK, linewidth=1.2)]
fig.legend(handles, ["Gaussian, matched mean/sd", r"$\mu_*$ (contamination centre)"],
           loc="lower center", ncol=2, frameon=False, fontsize=9, labelcolor=MUTED)

fig.suptitle(f"Simulated Y: n={n}, p={p}, contaminated normal "
             f"($\\mu_*$={mu_star:.2f})", fontsize=12, color=INK)
fig.tight_layout(rect=[0.02, 0.03, 1, 0.95])

# How much mass sits above mu_star beyond what a matched Gaussian predicts?
# (The uncontaminated body reaches past mu_star on its own, so the raw
# fraction above mu_star is NOT the contamination rate -- the excess is.)
obs = (Y > mu_star).mean()
exp = norm.sf(mu_star, Y.mean(), Y.std(ddof=1))
print(f"mu_star = {mu_star:.3f}")
print(f"  P(Y > mu_star): observed {obs:.3%}, matched Gaussian {exp:.3%}, "
      f"excess {obs - exp:+.3%}  (eps = 2%, half of it lands below mu_star)")
print(f"  pooled skew {skew(Y.ravel()):+.3f}   max |Y| {np.abs(Y).max():.2f}")


In [ ]:
# results = em.run_em_diagonal(Y, n_iter = 60, rho = rho_grid[5])